# Aging analysis – Priority Scheduling
Sinh 5 biểu đồ báo cáo từ 100 CSV bằng chính thuật toán của project.

In [ ]:
from pathlib import Path
import sys,csv
from statistics import mean
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if not (ROOT/"module7_fcfs.py").exists(): ROOT=Path(r"D:\HDH\Priority")
sys.path.insert(0,str(ROOT))
from module2_nonpreemptive import mo_phong_priority_nonpreemptive as np_run
from module3_preemptive import mo_phong_priority_preemptive as p_run
from module7_fcfs import mo_phong_fcfs as fcfs_run
DATA=ROOT/"data"/"default_dataset"
OUT=ROOT/"output"/"report_figures"; OUT.mkdir(parents=True,exist_ok=True)
print(ROOT,DATA,OUT,sep="\n")


In [ ]:
def load(path):
    with path.open(encoding="utf-8",newline="") as f:
        return [{"PID":r["PID"],"AT":int(r["AT"]),"BT":int(r["BT"]),"PR":int(r["PR"])} for r in csv.DictReader(f)]
def metrics(result):
    ps=result["processes"]; low=[p["WT"] for p in ps if 7<=p["PR"]<=9]
    return {"wt":result["average_waiting"],"tat":result["average_turnaround"],"rt":result["average_response"],
            "ctx":result["context_switches"],"maxwt":max(p["WT"] for p in ps),"lowwt":mean(low)}
files=sorted(DATA.glob("*.csv"))
assert len(files)==100,len(files)
datasets=[(p.name,load(p)) for p in files]
print("CSV:",len(datasets),"processes:",sum(len(x) for _,x in datasets))


In [ ]:
raw={"FCFS":[],"NP0":[],"P0":[]}
for k in range(1,11): raw[f"NP{k}"]=[]; raw[f"P{k}"]=[]
for i,(name,ps) in enumerate(datasets,1):
    raw["FCFS"].append(metrics(fcfs_run(ps)))
    raw["NP0"].append(metrics(np_run(ps))); raw["P0"].append(metrics(p_run(ps)))
    for k in range(1,11):
        raw[f"NP{k}"].append(metrics(np_run(ps,k)))
        raw[f"P{k}"].append(metrics(p_run(ps,k)))
    if i%10==0: print(f"{i}/100")
def avg(key,m): return mean(x[m] for x in raw[key])
S={key:{m:avg(key,m) for m in ("wt","tat","rt","ctx","maxwt","lowwt")} for key in raw}
for key in ("FCFS","NP0","P0"): print(key,{m:round(v,2) for m,v in S[key].items()})


In [ ]:
# 1) aging_baseline_detail.png
keys=["FCFS","NP0","P0"]; labels=["FCFS","Priority NP (No Aging)","Priority P (No Aging)"]; w=.24
fig,ax=plt.subplots(1,2,figsize=(13,6)); fig.suptitle("Baseline comparison without aging (100 CSV datasets)",fontweight="bold")
for a,metrics_,titles in [(ax[0],["wt","rt"],["Average WT","Average RT"]),(ax[1],["maxwt","lowwt"],["Max WT","Average WT (PR 7-9)"])]:
    x=list(range(len(metrics_)))
    for j,key in enumerate(keys):
        bars=a.bar([v+(j-1)*w for v in x],[S[key][m] for m in metrics_],w,label=labels[j]); a.bar_label(bars,fmt="%.2f",fontsize=8)
    a.set_xticks(x,titles); a.grid(axis="y",alpha=.25)
ax[0].set_title("Performance"); ax[1].set_title("Fairness")
h,l=ax[0].get_legend_handles_labels(); fig.legend(h,l,loc="lower center",ncol=3); fig.tight_layout(rect=(0,.08,1,.94))
fig.savefig(OUT/"aging_baseline_detail.png",dpi=220,bbox_inches="tight"); plt.show()


In [ ]:
# 2) aging_average_metrics_detail.png
ks=list(range(1,11)); fig,axes=plt.subplots(3,1,figsize=(12,11),sharex=True)
fig.suptitle("Average metrics vs aging interval k (100 CSV datasets)",fontweight="bold")
for a,(m,title) in zip(axes,[("wt","Average WT"),("tat","Average TAT"),("rt","Average RT")]):
    a.plot(ks,[S["FCFS"][m]]*10,"o--",label="FCFS baseline")
    a.plot(ks,[S[f"NP{k}"][m] for k in ks],"s-",label="Priority NP + Aging")
    a.plot(ks,[S[f"P{k}"][m] for k in ks],"^-",label="Priority P + Aging")
    a.set_title(title); a.grid(alpha=.25); a.legend()
axes[-1].set_xticks(ks); axes[-1].set_xlabel("Aging interval k")
fig.tight_layout(rect=(0,0,1,.95)); fig.savefig(OUT/"aging_average_metrics_detail.png",dpi=220,bbox_inches="tight"); plt.show()


In [ ]:
# 3) aging_fairness_detail.png
fig,ax=plt.subplots(1,2,figsize=(14,6)); fig.suptitle("Fairness metrics vs aging interval k (100 CSV datasets)",fontweight="bold")
for a,m,title in [(ax[0],"maxwt","Max WT"),(ax[1],"lowwt","Average WT of PR 7-9")]:
    a.plot(ks,[S["FCFS"][m]]*10,"o-",label="FCFS baseline")
    a.plot(ks,[S["NP0"][m]]*10,"o-",label="Priority NP No Aging")
    a.plot(ks,[S["P0"][m]]*10,"o-",label="Priority P No Aging")
    a.plot(ks,[S[f"NP{k}"][m] for k in ks],"s-",label="Priority NP + Aging")
    a.plot(ks,[S[f"P{k}"][m] for k in ks],"^-",label="Priority P + Aging")
    a.set_title(title); a.set_xticks(ks); a.grid(alpha=.25); a.legend(fontsize=8)
fig.tight_layout(rect=(0,0,1,.94)); fig.savefig(OUT/"aging_fairness_detail.png",dpi=220,bbox_inches="tight"); plt.show()


In [ ]:
# 4) aging_win_rate_detail.png
def win(candidate,base,m): return 100*sum(a[m]<b[m] for a,b in zip(raw[candidate],raw[base]))/100
np_fc=[win(f"NP{k}","FCFS","wt") for k in ks]; p_fc=[win(f"P{k}","FCFS","wt") for k in ks]
np_max=[win(f"NP{k}","NP0","maxwt") for k in ks]; p_max=[win(f"P{k}","P0","maxwt") for k in ks]
np_low=[win(f"NP{k}","NP0","lowwt") for k in ks]; p_low=[win(f"P{k}","P0","lowwt") for k in ks]
fig,ax=plt.subplots(1,2,figsize=(14,6)); fig.suptitle("Win rates vs aging interval k (100 CSV datasets)",fontweight="bold")
ax[0].plot(ks,np_fc,"o-",label="Priority NP + Aging"); ax[0].plot(ks,p_fc,"s-",label="Priority P + Aging")
for vals,label,mark in [(np_max,"NP Aging vs NP No Aging: Max WT","o-"),(p_max,"P Aging vs P No Aging: Max WT","s-"),(np_low,"NP Aging vs NP No Aging: WT PR 7-9","^-"),(p_low,"P Aging vs P No Aging: WT PR 7-9","d-")]: ax[1].plot(ks,vals,mark,label=label)
for a,title in zip(ax,["Aging beats FCFS on Average WT","Aging beats no-aging on fairness"]):
    a.set_title(title); a.set_xticks(ks); a.set_ylim(0,105); a.set_ylabel("Win rate (%)"); a.grid(alpha=.25); a.legend(fontsize=8)
fig.tight_layout(rect=(0,0,1,.94)); fig.savefig(OUT/"aging_win_rate_detail.png",dpi=220,bbox_inches="tight"); plt.show()
print("NP vs FCFS:",np_fc,"\nP vs FCFS:",p_fc,"\nlow priority:",np_low,p_low)


In [ ]:
# 5) starvation_pressure_detail.png
idx=[i for i,(name,_) in enumerate(datasets) if "starvation_pressure" in name]; assert len(idx)==20
def sub(key,m): return mean(raw[key][i][m] for i in idx)
keys=["FCFS","NP0","NP1","NP10","P0","P1","P10"]; labels=["FCFS","NP No Aging","NP k=1","NP k=10","P No Aging","P k=1","P k=10"]
x=list(range(7)); w=.23; fig,ax=plt.subplots(figsize=(14,7)); fig.suptitle("Starvation pressure scenario (20 CSV datasets)",fontweight="bold")
for shift,m,label in [(-w,"wt","Average WT"),(0,"maxwt","Max WT"),(w,"lowwt","Average WT (PR 7-9)")]:
    bars=ax.bar([v+shift for v in x],[sub(k,m) for k in keys],w,label=label); ax.bar_label(bars,fmt="%.2f",fontsize=8)
ax.set_xticks(x,labels); ax.grid(axis="y",alpha=.25); ax.legend()
fig.tight_layout(rect=(0,0,1,.94)); fig.savefig(OUT/"starvation_pressure_detail.png",dpi=220,bbox_inches="tight"); plt.show()


In [ ]:
expected=["aging_baseline_detail.png","aging_average_metrics_detail.png","aging_fairness_detail.png","aging_win_rate_detail.png","starvation_pressure_detail.png"]
for name in expected: print(name,(OUT/name).exists(),OUT/name)
